In [ ]:
import os
import gc
import sys
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib import colors
import matplotlib.ticker as mticker

In [ ]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

In [ ]:
# Path of PLUMBER 2 dataset
PLUMBER2_path      = "/g/data/w97/mm3972/data/PLUMBER2/"
PLUMBER2_flux_path = "/g/data/w97/mm3972/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
PLUMBER2_met_path  = "/g/data/w97/mm3972/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"
site_names, IGBP_types, clim_types, model_names = load_default_list()

## Plot for each site

In [ ]:
ordered_colors ={0:'red', 1: 'darkorange',2:'orange',3:'gold',4:'yellowgreen',5:'green',6:'mediumseagreen',
               7:'lime',8:'aquamarine',9:'cyan',10:'dodgerblue',11:'blue',12:'darkolivegreen',
               13:'forestgreen',14:'lime',15:'gold', 16:'orange',17:'pink',18:'pink',19:'red',20:'deeppink',
               21:'mediumorchid',22: 'darkviolet',}

In [ ]:
model_colors = {
                'obs': 'black',
                'obs_cor': 'dimgrey',
                '1lin': 'lightcoral' ,
                '3km27': 'indianred',
                '6km729': 'firebrick',
                '6km729lag':'red',
                'LSTM_eb': 'coral',
                'LSTM_raw': 'pink',
                'RF_eb': 'tomato',
                'RF_raw': 'deeppink',
                'Manabe':'violet',
                'ManabeV2':'darkviolet',
                'PenmanMonteith': 'purple',
                'CABLE':'darkblue',
                'CABLE-POP-CN':'blue',
                'CHTESSEL_ERA5_3':'cornflowerblue',
                'CHTESSEL_Ref_exp1':'dodgerblue',
                'CLM5a':'deepskyblue',
                'GFDL':'c',
                'JULES_GL9':'limegreen', 
                'JULES_GL9_withLAI':'aquamarine',
                'JULES_test':'lightseagreen',
                'MATSIRO':'darkcyan',
                'NoahMPv401':'darkolivegreen',
                'ORC2_r6593':'forestgreen',
                'ORC2_r6593_CO2':'limegreen',
                'ORC3_r7245_NEE':'lime',
                'ORC3_r8120':'lightgreen',
                'STEMMUS-SCOPE':'yellowgreen',
                'ACASA':'yellow',
                'LPJ-GUESS':'orange',
                'MuSICA':'gold',
                'NASAEnt': 'goldenrod',
                'QUINCY':'peru',
                'SDGVM':'sandybrown',
                }


<h3 style="color:blue;">Get model list</h3>  

In [ ]:
PLUMBER2_path_site = "/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/AU-How.nc"
f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
model_list         = f.variables['NEE_models'][:]
model_list         = model_list.tolist()
model_list.append('obs')

In [ ]:
model_list

In [ ]:
time       = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,only_use_cftime_datetimes=False,only_use_python_datetimes=True)
ntime      = len(time)
month      = np.zeros(ntime)
hour       = np.zeros(ntime)

for i,t in enumerate(time):
    month[i] = t.month
    hour[i]  = t.hour #+0.25#+ t.minute/60.

<h3 style="color:blue;">Check monthly diurnal cycle</h3>

#### Obs

In [ ]:
model_in      = 'obs'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables[model_in+'_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### 1lin

In [ ]:
model_in      = '1lin'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### 3km27

In [ ]:
model_in      = '3km27'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### 6km729

In [ ]:
model_in      = '6km729'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### 6km729lag

In [ ]:
model_in      = '6km729lag'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### RF_eb

In [ ]:
model_in      = 'RF_eb'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### RF_raw

In [ ]:
model_in      = 'RF_raw'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### LSTM_eb

In [ ]:
model_in      = 'LSTM_eb'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### LSTM_raw

In [ ]:
model_in      = 'LSTM_raw'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)
    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### CABLE

In [ ]:
model_in      = 'CABLE'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### CABLE-POP-CN

In [ ]:
model_in      = 'CABLE-POP-CN'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### GFDL

In [ ]:
model_in      = 'GFDL'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### Noah MP

In [ ]:
model_in      = 'NoahMPv401'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### MuSICA

In [ ]:
model_in      = 'MuSICA'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### STEMMUS-SCOPE

In [ ]:
model_in      = 'STEMMUS-SCOPE'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    else:
        sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### JULES_GL9

In [ ]:
model_in      = 'JULES_GL9'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_GPP'][:].data, columns=['GPP'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour


ax2 = ax.twinx()
ax3 = ax.twinx()

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

    
    sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))
    
#     if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
#         sct = ax.plot(var_diurnal_cycle['NEE'], lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
#     else:
#         sct = ax.plot(var_diurnal_cycle['NEE']*(-1), lw=2.0, color=model_colors[m], alpha=0.6, ls='--', label=str(m))
    
    # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
    sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[m], alpha=0.3, ls=':')
    sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[m], alpha=0.1, ls='--')
    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)
    
    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.00001, 0.0003)
    ax2.set_ylim(-30, 900)
    ax3.set_ylim(-0.15, 4.5)

plt.show()

#### all models in one plot

In [ ]:
# for site_name in site_names:

site_name = "AU-Tum"

PLUMBER2_path_site = "/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/"+site_name+".nc"
f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
model_list         = f.variables['NEE_models'][:]
model_list         = model_list.tolist()
model_list.append('obs')

time       = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,only_use_cftime_datetimes=False,only_use_python_datetimes=True)
ntime      = len(time)
month      = np.zeros(ntime)
hour       = np.zeros(ntime)

for i,t in enumerate(time):
    month[i] = t.month
    hour[i]  = t.hour #+0.25#+ t.minute/60.

fig, ax       = plt.subplots(nrows=3, ncols=4, figsize=[24, 20])
for i, model_in in enumerate(model_list):

    var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
    var['Qle']    = f.variables[model_in+'_Qle'][:].data
    # var['GPP']    = f.variables[model_in+'_GPP'][:].data
    var['SWdown'] = f.variables['obs_SWdown'][:].data
    var['VPD']    = f.variables['VPD'][:].data
    var['month']  = month
    var['hour']   = hour

    for m in np.arange(12):

        row = m//4
        col = m%4

        var_masked        = var[month == m+1]
        var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)

        # ax2 = ax[row,col].twinx()
        # ax3 = ax[row,col].twinx()

        # sct = ax.plot(var_diurnal_cycle['GPP'], lw=2.0, color=model_colors[m], alpha=0.9, label=str(m))

        if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
            sct = ax[row,col].plot(var_diurnal_cycle.loc[var_diurnal_cycle['SWdown']>50, 'NEE'], lw=2.0, color=model_colors[model_in], alpha=0.9, label=str(model_in)) # , ls='--'
        else:
            sct = ax[row,col].plot(var_diurnal_cycle.loc[var_diurnal_cycle['SWdown']>50, 'NEE']*(-1), lw=2.0, color=model_colors[model_in], alpha=0.9, label=str(model_in)) # , ls='--'

        # sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
        # sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=2.0, color=model_colors[i], alpha=0.3, ls=':')
        # sct = ax3.plot(var_diurnal_cycle['VPD'], lw=2.0, color=model_colors[i], alpha=0.1, ls='--')
        # ax[row,col].set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)

        ax[row,col].set_ylim(-0.0001, 0.0004)
        # ax2.set_ylim(-30, 900)
        # ax3.set_ylim(-0.15, 4.5)
        if row == 0 and col ==0:
            ax[row,col].legend(fontsize=8,frameon=False)
        # if col == 3:
        #     ax3.spines["right"].set_position(("outward", 50))
    
fig.savefig("./plots/Diurnal_Cycle_NEE_SWdown_"+site_name,bbox_inches='tight',dpi=300)
ax=None

<h3 style="color:blue;">Check diurnal hysteresis</h3> 

#### Obs

In [ ]:
model_in      = 'GFDL'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
# var           = pd.DataFrame(f.variables[model_in+'_GPP'][:].data, columns=['GPP'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
# var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour

if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
    var.loc[:,'NEE'] = var['NEE'][:]
else:
    var.loc[:,'NEE'] = var['NEE'][:]*(-1)

var_name1     = "SWdown"
var_name2     = "NEE" 

for m in [0,1,2,9,10,11]:
# for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)
    
    var2_mean              = np.mean(var_diurnal_cycle[var_name2])
    var2_diurnal_cycle_std = var_diurnal_cycle[var_name2]/var2_mean
    ax.scatter(var_diurnal_cycle[var_name1],var2_diurnal_cycle_std, c=model_colors[m], label=str(m+1))
    
    for i in np.arange(1,24):  # Start from 1 to access the previous point
        ax.annotate("",
                    xy=(var_diurnal_cycle[var_name1][i], var2_diurnal_cycle_std[i]),
                    xytext=(var_diurnal_cycle[var_name1][i-1], var2_diurnal_cycle_std[i-1]),
                    arrowprops=dict(arrowstyle="->", color=model_colors[m]))

    ax.legend(fontsize=8,frameon=False)
    # ax.set_ylim(-0.0001, 0.00025)

plt.show()
var = None

Caution: when use day mean to divide the diurnal cycle, cold season with low GPP or NEE may have very large hysteresis cycles. 

#### CABLE

In [ ]:
model_in      = 'obs'
fig, ax       = plt.subplots(figsize=[10, 7])

var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['GPP']    = f.variables[model_in+'_GPP'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour

if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
    var.loc[:,'NEE'] = var['NEE'][:]
else:
    var.loc[:,'NEE'] = var['NEE'][:]*(-1)

var_name1     = "SWdown"
var_name2     = "NEE" 


for m in [0,1,2,3,4,8,9,10,11]:
# for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)
    
    ax.scatter(var_diurnal_cycle[var_name1],var_diurnal_cycle[var_name2], c=model_colors[m], label=str(m+1))
    
    for i in np.arange(1,24):  # Start from 1 to access the previous point
        ax.annotate("",
                    xy=(var_diurnal_cycle[var_name1][i], var_diurnal_cycle[var_name2][i]),
                    xytext=(var_diurnal_cycle[var_name1][i-1], var_diurnal_cycle[var_name2][i-1]),
                    arrowprops=dict(arrowstyle="->", color=model_colors[m]))

    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.0001, 0.00025)

plt.show()
var = None

#### LSTM_raw

In [ ]:
model_in      = 'LSTM_raw'
fig, ax       = plt.subplots(figsize=[10, 7])
try:
    var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
except:
    var           = pd.DataFrame(f.variables[model_in+'_GPP'][:].data, columns=['GPP'])
    
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour

if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
    var.loc[:,'NEE'] = var['NEE'][:]
else:
    var.loc[:,'NEE'] = var['NEE'][:]*(-1)

var_name1     = "SWdown"
var_name2     = "NEE" 

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)
    
    ax.scatter(var_diurnal_cycle[var_name1],var_diurnal_cycle[var_name2], c=model_colors[m], label=str(m+1))
    
    for i in np.arange(1,24):  # Start from 1 to access the previous point
        ax.annotate("",
                    xy=(var_diurnal_cycle[var_name1][i], var_diurnal_cycle[var_name2][i]),
                    xytext=(var_diurnal_cycle[var_name1][i-1], var_diurnal_cycle[var_name2][i-1]),
                    arrowprops=dict(arrowstyle="->", color=model_colors[m]))

    ax.legend(fontsize=8,frameon=False)
    ax.set_ylim(-0.0001, 0.00025)

plt.show()
var = None

#### JULES_GL9

In [ ]:
model_in      = 'JULES_GL9'
fig, ax       = plt.subplots(figsize=[10, 7])
try:
    var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
        var.loc[:,'NEE'] = var['NEE'][:]
    else:
        var.loc[:,'NEE'] = var['NEE'][:]*(-1)
    var['GPP']    = f.variables[model_in+'_GPP'][:].data
except:
    var       = pd.DataFrame(f.variables[model_in+'_GPP'][:].data, columns=['GPP'])
    
var['Qle']    = f.variables[model_in+'_Qle'][:].data
var['SWdown'] = f.variables['obs_SWdown'][:].data
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour

var_name1     = "SWdown"
var_name2     = "Qle"

for m in np.arange(12):
    var_masked        = var[month == m+1]
    var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)
    
    ax.scatter(var_diurnal_cycle[var_name1],var_diurnal_cycle[var_name2], c=model_colors[m], label=str(m+1))
    
    for i in np.arange(1,24):  # Start from 1 to access the previous point
        ax.annotate("",
                    xy=(var_diurnal_cycle[var_name1][i], var_diurnal_cycle[var_name2][i]),
                    xytext=(var_diurnal_cycle[var_name1][i-1], var_diurnal_cycle[var_name2][i-1]),
                    arrowprops=dict(arrowstyle="->", color=model_colors[m]))

    ax.legend(fontsize=8,frameon=False)
    # ax.set_ylim(-0.0001, 0.00025)

plt.show()
var = None

#### all models in one plot

In [ ]:
for site_name in site_names:
    
    PLUMBER2_path_site = "/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/"+site_name+".nc"
    f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
    model_list         = f.variables['NEE_models'][:]
    model_list         = model_list.tolist()
    model_list.append('obs')
    
    time       = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,only_use_cftime_datetimes=False,only_use_python_datetimes=True)
    ntime      = len(time)
    month      = np.zeros(ntime)
    hour       = np.zeros(ntime)

    for i,t in enumerate(time):
        month[i] = t.month
        hour[i]  = t.hour #+0.25#+ t.minute/60.

    fig, ax = plt.subplots(nrows=3, ncols=4, figsize=[20, 20])

    for i, model_in in enumerate(model_list):
        # print(model_in)
        try:
            var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
            if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
                var.loc[:,'NEE'] = var['NEE'][:]
            else:
                var.loc[:,'NEE'] = var['NEE'][:]*(-1)
                # var['GPP']= f.variables[model_in+'_GPP'][:].data
        except:
            print('no NEE')
            # var       = pd.DataFrame(f.variables[model_in+'_GPP'][:].data, columns=['GPP'])

        var['Qle']    = f.variables[model_in+'_Qle'][:].data
        var['SWdown'] = f.variables['obs_SWdown'][:].data
        var['VPD']    = f.variables['VPD'][:].data
        var['month']  = month
        var['hour']   = hour #+ 0.25

        var_name1     = "SWdown"
        var_name2     = "NEE"

        for m in np.arange(12):
            # print(m+1)
            row = m//4
            col = m%4

            var_masked        = var[month == m+1]
            var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)
            # print(len(var_diurnal_cycle[var_name1]))
            ax[row,col].scatter(var_diurnal_cycle[var_name1],var_diurnal_cycle[var_name2], c=model_colors[model_in], label=str(model_in))

            for k in np.arange(1,24):  # Start from 1 to access the previous point
                ax[row,col].annotate("",
                            xy=(var_diurnal_cycle[var_name1][k], var_diurnal_cycle[var_name2][k]),
                            xytext=(var_diurnal_cycle[var_name1][k-1], var_diurnal_cycle[var_name2][k-1]),
                            arrowprops=dict(arrowstyle="->", color=model_colors[model_in]))

            # ax[0,0].legend(fontsize=8,frameon=False)
            ax[row,col].set_ylim(-0.0001, 0.00025)


    fig.savefig("./plots/Hysteresis_NEE_SWdown_"+site_name,bbox_inches='tight',dpi=300)
    ax=None


#### all models in one plot (normalized by max-min)

In [ ]:
for site_name in site_names:
    
    PLUMBER2_path_site = "/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/"+site_name+".nc"
    f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
    model_list         = f.variables['NEE_models'][:]
    model_list         = model_list.tolist()
    model_list.append('obs')
    
    time       = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,only_use_cftime_datetimes=False,only_use_python_datetimes=True)
    ntime      = len(time)
    month      = np.zeros(ntime)
    hour       = np.zeros(ntime)

    for i,t in enumerate(time):
        month[i] = t.month
        hour[i]  = t.hour #+0.25#+ t.minute/60.

    fig, ax = plt.subplots(nrows=3, ncols=4, figsize=[18, 15])

    # for i, model_in in enumerate(model_list):
    model_in = 'obs'
    try:
        var           = pd.DataFrame(f.variables[model_in+'_NEE'][:].data, columns=['NEE'])
        if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
            var.loc[:,'NEE'] = var['NEE'][:]
        else:
            var.loc[:,'NEE'] = var['NEE'][:]*(-1)
            # var['GPP']= f.variables[model_in+'_GPP'][:].data
    except:
        print('no NEE')
        # var       = pd.DataFrame(f.variables[model_in+'_GPP'][:].data, columns=['GPP'])

    var['Qle']    = f.variables[model_in+'_Qle'][:].data
    var['SWdown'] = f.variables['obs_SWdown'][:].data
    var['VPD']    = f.variables['VPD'][:].data
    var['month']  = month
    var['hour']   = hour #+ 0.25

    var_name1     = "SWdown"
    var_name2     = "NEE"

    for m in np.arange(12):

        row = m//4
        col = m%4

        var_masked        = var[month == m+1]
        var_diurnal_cycle = var_masked.groupby(['hour']).mean(numeric_only=True)
        var_diurnal_cycle[var_name2] = var_diurnal_cycle[var_name2]/(np.max(var_diurnal_cycle[var_name2])-np.min(var_diurnal_cycle[var_name2]))

        # print(len(var_diurnal_cycle[var_name1]))
        ax[row,col].scatter(var_diurnal_cycle[var_name1], var_diurnal_cycle[var_name2], c=model_colors[model_in], label=str(model_in))

        for k in np.arange(1,24):  # Start from 1 to access the previous point
            ax[row,col].annotate("",
                        xy=(var_diurnal_cycle[var_name1][k], var_diurnal_cycle[var_name2][k]),
                        xytext=(var_diurnal_cycle[var_name1][k-1], var_diurnal_cycle[var_name2][k-1]),
                        arrowprops=dict(arrowstyle="->", color=model_colors[model_in]))

        # ax[0,0].legend(fontsize=8,frameon=False)
        # ax[row,col].set_ylim(-0.0001, 0.00025)
    
    print(f"Figure size: {fig.get_size_inches()}")

    fig.savefig("./plots/Hysteresis_normalized_NEE_SWdown_"+site_name,bbox_inches='tight',dpi=100)
    
    f.close()
    # fig = None
    gc.collect()
    ax  = None


<h3 style="color:blue;">Seasonal cycle</h3>   

In [ ]:
fig, ax            = plt.subplots(figsize=[10, 7])

PLUMBER2_path_site = "/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/AU-How.nc"
f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
model_list         = f.variables['GPP_models'][:]
model_list         = model_list.tolist()
model_list.append('obs')

time       = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,only_use_cftime_datetimes=False,only_use_python_datetimes=True)
ntime      = len(time)
month      = np.zeros(ntime)
hour       = np.zeros(ntime)

for i,t in enumerate(time):
    month[i] = t.month
    hour[i]  = t.hour #+0.25#+ t.minute/60.
    
var           = pd.DataFrame(f.variables['obs_SWdown'][:].data, columns=['SWdown'])
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour

for model_in in model_list:    
    # var[model_in+'_NEE']    = f.variables[model_in+'_NEE'][:].data
    var[model_in+'_Qle']    = f.variables[model_in+'_Qle'][:].data
    var[model_in+'_GPP']    = f.variables[model_in+'_GPP'][:].data

    
var_diurnal_cycle = var.groupby(['month']).mean(numeric_only=True)
ax2 = ax.twinx()
ax3 = ax.twinx()

# sct = ax2.plot(var_diurnal_cycle['Qle'], lw=2.0, color=model_colors[m], alpha=0.6, ls='-.')
sct = ax2.plot(var_diurnal_cycle['SWdown'], lw=3.0, color='black', alpha=1, ls=':')
sct = ax3.plot(var_diurnal_cycle['VPD'], lw=3.0, color='black', alpha=1, ls='--')

for model_in in model_list:   
    sct = ax.plot(var_diurnal_cycle[model_in+'_GPP'], lw=2.0, color=model_colors[model_in], alpha=0.9, label=model_in)

#     if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
#         sct = ax.plot(var_diurnal_cycle[model_in+'_NEE'], lw=2.0, color=model_colors[model_in], alpha=0.6, ls='-',label=model_in)
#     else:
#         sct = ax.plot(var_diurnal_cycle[model_in+'_NEE']*(-1), lw=2.0, color=model_colors[model_in], alpha=0.6, ls='-',label=model_in)

    ax3.spines["right"].set_position(("outward", 50))
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)

    ax.legend(fontsize=8,frameon=False)
    # ax.set_ylim(-0.00001, 0.0003)
    # ax2.set_ylim(-30, 900)
    # ax3.set_ylim(-0.15, 4.5)

plt.show()

<h3 style="color:blue;">Season hysteresis</h3> 

In [ ]:
fig, ax            = plt.subplots(figsize=[10, 7])

PLUMBER2_path_site = "/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/AU-Tum.nc"
f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
model_list         = f.variables['NEE_models'][:]
model_list         = model_list.tolist()
model_list.append('obs')

time       = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,only_use_cftime_datetimes=False,only_use_python_datetimes=True)
ntime      = len(time)
month      = np.zeros(ntime)
hour       = np.zeros(ntime)

for i,t in enumerate(time):
    month[i] = t.month
    hour[i]  = t.hour #+0.25#+ t.minute/60.
    
var           = pd.DataFrame(f.variables['obs_SWdown'][:].data, columns=['SWdown'])
var['VPD']    = f.variables['VPD'][:].data
var['month']  = month
var['hour']   = hour

for model_in in model_list:    

    try:
        if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
            var.loc[:,model_in+'_NEE'] = f.variables[model_in+'_NEE'][:].data
        else:
            var.loc[:,model_in+'_NEE'] = f.variables[model_in+'_NEE'][:].data*(-1)
            # var['GPP']= f.variables[model_in+'_GPP'][:].data
    except:
        print('no NEE')
        
var_diurnal_cycle = var.groupby(['month']).mean(numeric_only=True)

for model_in in model_list[8:]:   
    sct = ax.scatter(var_diurnal_cycle['SWdown'], var_diurnal_cycle[model_in+'_NEE'], color=model_colors[model_in], alpha=0.9, label=model_in)

    for k in np.arange(1,12):  # Start from 1 to access the previous point
        ax.annotate("",
                    xy=(var_diurnal_cycle.iloc[k]['SWdown'], var_diurnal_cycle.iloc[k][model_in+'_NEE']),
                    xytext=(var_diurnal_cycle.iloc[k-1]['SWdown'], var_diurnal_cycle.iloc[k-1][model_in+'_NEE']),
                    arrowprops=dict(arrowstyle="->", color=model_colors[model_in]))
        
    ax.set_ylabel("Net ecosystem exchange of CO2 (g C m$\mathregular{^{-1}}$ h$\mathregular{^{-1}}$)", fontsize=12)

    ax.legend(fontsize=8,frameon=False)
    # ax.set_ylim(-0.00001, 0.0003)
    # ax2.set_ylim(-30, 900)
    # ax3.set_ylim(-0.15, 4.5)

plt.show()

<h3 style="color:blue;">Season hysteresis</h3> 